### Import Library

In [ ]:
import pandas as pd
import numpy as np
import random
import re
import copy
import warnings
import sys
import traceback

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:

def custom_formatwarning(msg, *args, **kwargs):
    return f"Warning: {msg}\n"

warnings.formatwarning = custom_formatwarning

### Text Preprocessing Pipeline

In [4]:
def text_clean_pipeline(text: str) -> str:
    """Replicate the same Text_Clean pipeline used on labeled data."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'&[a-z]+;', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### Load Clean Dataset

In [5]:
df = pd.read_csv("../dataset/Clean.csv")

print(df.columns)
df.head()

Index(['No', 'answer', '1-FR', '2-GI', '3-PI', '4-DM', '5-EDTRB', '6-RE',
       'Text_Clean', 'filtered_text', 'token', 'tokens_stemmed',
       'Process_Data'],
      dtype='str')


,No,answer,1-FR,2-GI,3-PI,4-DM,5-EDTRB,6-RE,Text_Clean,filtered_text,token,tokens_stemmed,Process_Data
0,1,"Halo Rizal,Radang tenggorokan umunya disebabka...",1,0,1,1,1,0,halo rizal radang tenggorokan umunya disebabka...,halo rizal radang tenggorokan umunya disebabka...,"['halo', 'rizal', 'radang', 'tenggorokan', 'um...","['halo', 'rizal', 'radang', 'tenggorok', 'umu'...",halo rizal radang tenggorok umu sebab infeksi ...
1,2,"Halo Hellas,Cacar air merupakan suatu penyakit...",1,0,1,1,1,0,halo hellas cacar air merupakan suatu penyakit...,halo hellas cacar air penyakit disebabkan viru...,"['halo', 'hellas', 'cacar', 'air', 'penyakit',...","['halo', 'hellas', 'cacar', 'air', 'sakit', 's...",halo hellas cacar air sakit sebab virus varise...
2,3,Halo Rory.......Terimakasih atas pertanyaan An...,1,0,1,1,1,0,halo rory terimakasih atas pertanyaan anda per...,halo rory terimakasih ketahui gangguan kulit s...,"['halo', 'rory', 'terimakasih', 'ketahui', 'ga...","['halo', 'rory', 'terimakasih', 'tahu', 'gangg...",halo rory terimakasih tahu ganggu kulit rangka...
3,4,"Alo AfriYani, Terimakasih atas pertanyaannya. ...",1,0,1,1,1,0,alo afriyani terimakasih atas pertanyaannya ku...,alo afriyani terimakasih pertanyaannya kuku ja...,"['alo', 'afriyani', 'terimakasih', 'pertanyaan...","['alo', 'afriyani', 'terimakasih', 'tanya', 'k...",alo afriyani terimakasih tanya kuku jari kaki ...
4,5,"Halo,Telinga berdenging atau tinitus merupak...",1,0,1,1,1,0,halo telinga berdenging atau tinitus merupakan...,halo telinga berdenging tinitus sensasi penden...,"['halo', 'telinga', 'berdenging', 'tinitus', '...","['halo', 'telinga', 'denging', 'tinitus', 'sen...",halo telinga denging tinitus sensasi dengar de...


### Prepare Data

In [6]:
TEXT_COL = "Text_Clean"  

LABEL_COLS = [
    "1-FR",
    "2-GI",
    "3-PI",
    "4-DM",
    "5-EDTRB",
    "6-RE"
]

X = df[TEXT_COL]

y = df[LABEL_COLS].values

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (500,)
Shape y: (500, 6)


### Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [8]:
print("Distribusi label (train):")
print(pd.DataFrame(y_train, columns=LABEL_COLS).sum())

print("\nJumlah unik tiap label (train):")
print(pd.DataFrame(y_train, columns=LABEL_COLS).nunique())

Distribusi label (train):
1-FR       400
2-GI        37
3-PI       400
4-DM       353
5-EDTRB    396
6-RE        21
dtype: int64

Jumlah unik tiap label (train):
1-FR       1
2-GI       2
3-PI       1
4-DM       2
5-EDTRB    2
6-RE       2
dtype: int64


### TF-IDF + Model Training

In [ ]:
best_model = None
best_vectorizer = None
best_f1 = 0
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, class_weight='balanced')
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model.fit(X_train_tfidf, y_train)

### Evaluasi Awal

In [9]:
y_pred = model.predict(X_test_tfidf)
print("EVALUASI AWAL")

print("F1 Micro:", f1_score(y_test, y_pred, average='micro'))
print("F1 Macro:", f1_score(y_test, y_pred, average='macro'))

print(classification_report(y_test, y_pred, target_names=LABEL_COLS))
baseline_f1 = f1_score(y_test, y_pred, average='macro')

best_model = copy.deepcopy(model)
best_vectorizer = copy.deepcopy(vectorizer)
best_f1 = baseline_f1

print("Baseline F1 Macro:", baseline_f1)

EVALUASI AWAL
F1 Micro: 0.9521276595744681
F1 Macro: 0.6895042377800998
              precision    recall  f1-score   support

        1-FR       1.00      1.00      1.00       100
        2-GI       0.50      0.25      0.33         4
        3-PI       1.00      1.00      1.00       100
        4-DM       1.00      0.69      0.81        86
     5-EDTRB       0.99      0.99      0.99        99
        6-RE       0.00      0.00      0.00         3

   micro avg       0.99      0.91      0.95       392
   macro avg       0.75      0.65      0.69       392
weighted avg       0.98      0.91      0.94       392
 samples avg       0.99      0.92      0.95       392

Baseline F1 Macro: 0.6895042377800998


### Load Raw Dataset

In [10]:
raw_df = pd.read_csv("../dataset/Raw.csv")

print(raw_df.columns)
raw_df.head()

Index(['title', 'question', 'question_date', 'answer', 'answer_date', 'topics',
       'topic_set', 'risk', 'year', 'time_to_answer'],
      dtype='str')


,title,question,question_date,answer,answer_date,topics,topic_set,risk,year,time_to_answer
0,Khasiat obat zinc sulphate,Dok saya mau tanya Anak saya kan kenak fimosis...,"23 September 2017, 18:50","Hai Yevie, Terima kasih atas pertanyaannya. Zi...","24 September 2017, 10:42",zinc-sulphate,zinc-sulphate,low,2017,0.0
1,Perbedaan jenis formula zinc,siang dokter.... dokter sayang ingin bertanya ...,"5 August 2017, 12:16","Halo Pendys, Zinc merupakan salah satu minera...","5 August 2017, 16:27",zinc-sulphate,zinc-sulphate,low,2017,0.0
2,Mengkonsumsi suplemen zinc yang sudah kadaluarsa,"Malam dok, saya baru menemukan suplemen zinc s...","12 December 2018, 20:54","Hai IriSh, Terimakasih telah bertanya ke Alodo...","13 December 2018, 17:08",suplemen zinc-sulphate,zinc-sulphate,low,2018,0.0
3,Keamanan konsumsi suplemen zinc saat program h...,"Dear dokter, Umur saya 24 tahun dan baru menik...","7 January 2019, 15:09","Selamat pagi, terimakasih atas pertanyaannya S...","8 January 2019, 09:32",suplemen zinc-sulphate,zinc-sulphate,low,2019,0.0
4,Suplemen apa yang banyak mengandung zinc.,Sakit flu tak kunjung sembuh disebabkan karena...,"30 March 2019, 06:05","Selamat malam, terimakasih atas pertanyaannya ...","30 March 2019, 20:25",suplemen zinc-sulphate,zinc-sulphate,low,2019,0.0


In [ ]:
RAW_TEXT_COL = "answer"  

X_unlabeled = (
    raw_df[RAW_TEXT_COL]
    .dropna()
    .apply(text_clean_pipeline)
)
X_unlabeled = X_unlabeled[X_unlabeled.str.strip() != ""]

print("Jumlah raw data:", len(X_unlabeled))

Jumlah raw data: 497974


### Fungsi Pseudo Label

In [12]:
def get_high_confidence_predictions(model, X_tfidf, df, threshold=0.85):
    probs = model.predict_proba(X_tfidf)
    
    selected_indices = []
    pseudo_labels = []
    
    for i, prob in enumerate(probs):
        labels = (prob >= threshold).astype(int)
        
        if labels.sum() > 0:
            selected_indices.append(df.index[i])
            pseudo_labels.append(labels)
    
    return selected_indices, np.array(pseudo_labels)

### Iterative Training (Semi-Supervised)

In [ ]:
max_iterations = 5
add_per_iter = 15
threshold = 0.8

X_train_current = X_train.copy()
y_train_current = y_train.copy()

X_unlabeled_current = X_unlabeled.copy()

used_indices = set()

best_f1 = baseline_f1  

for iteration in range(max_iterations):
    print(f"\nITERATION {iteration+1}")
    X_unlabeled_tfidf = vectorizer.transform(X_unlabeled_current)
    idx, pseudo_y = get_high_confidence_predictions(
        model, X_unlabeled_tfidf, X_unlabeled_current, threshold
    )
    idx = [i for i in idx if i not in used_indices]
    
    if len(idx) == 0:
        print("Tidak ada data confident, stop.")
        break
    
    if len(idx) > add_per_iter:
        idx = random.sample(idx, add_per_iter)
    
    used_indices.update(idx)
    
    X_new = X_unlabeled_current.loc[idx]
    
    idx_map = [list(X_unlabeled_current.index).index(i) for i in idx]
    pseudo_y_selected = pseudo_y[idx_map]
    
    X_train_current = pd.concat([X_train_current, X_new])
    y_train_current = np.vstack([y_train_current, pseudo_y_selected])
    
    X_unlabeled_current = X_unlabeled_current.drop(idx)

    X_train_tfidf = vectorizer.transform(X_train_current)
    model.fit(X_train_tfidf, y_train_current)

    y_pred = model.predict(X_test_tfidf) 
    f1_micro = f1_score(y_test, y_pred, average='micro')
    f1_macro = f1_score(y_test, y_pred, average='macro')
    
    print("F1 Micro:", f1_micro)
    print("F1 Macro:", f1_macro)
    
    if f1_macro > best_f1:
        best_f1 = f1_macro
        best_model = copy.deepcopy(model)
        best_vectorizer = copy.deepcopy(vectorizer)
        print("Improved! Best F1 Macro:", best_f1)
    else:
        print("Performa menurun")


ITERATION 1


F1 Micro: 0.9293478260869565
F1 Macro: 0.6649831649831649
Performa menurun

ITERATION 2


F1 Micro: 0.9178082191780822
F1 Macro: 0.6514330294818099
Performa menurun

ITERATION 3


F1 Micro: 0.9042995839112344
F1 Macro: 0.6328194222931065
Performa menurun

ITERATION 4


F1 Micro: 0.9025069637883009
F1 Macro: 0.6289471289471289
Performa menurun

ITERATION 5
F1 Micro: 0.900976290097629
F1 Macro: 0.6265993265993266
Performa menurun


In [14]:
print("Best F1 Macro final:", best_f1)

Best F1 Macro final: 0.6895042377800998


In [15]:
X_sample = X_unlabeled.sample(n=10000, random_state=42) 

tfidf_all = best_vectorizer.transform(X_sample)
pred_all = best_model.predict(tfidf_all)

predictions = []

for pred in pred_all:
    labels = [LABEL_COLS[i] for i in range(len(pred)) if pred[i] == 1]
    predictions.append(labels)


sample_df = raw_df.loc[X_sample.index].copy()

sample_df["predicted_labels"] = predictions

print(sample_df[["answer", "predicted_labels"]].head())

                                                   answer  \
178403  Hai Rendi,Nyeri dada seperti yang Anda alami p...   
455608  Salam, Dari info yang Anda sampaikan, setelah ...   
189467  Halo,  Rasa linu / nyeri otot  yang dirasakan ...   
449623  Halo Andini...... Terimakasih atas pertanyaan ...   
108563  Halo,Pterigium merupakan selaput menonjol yang...   

                   predicted_labels  
178403  [1-FR, 3-PI, 4-DM, 5-EDTRB]  
455608        [1-FR, 3-PI, 5-EDTRB]  
189467  [1-FR, 3-PI, 4-DM, 5-EDTRB]  
449623  [1-FR, 3-PI, 4-DM, 5-EDTRB]  
108563        [1-FR, 3-PI, 5-EDTRB]  


In [16]:
sample_df = raw_df.loc[X_sample.index].copy()

sample_df["predicted_labels"] = predictions

print(sample_df[["answer", "predicted_labels"]].head())

                                                   answer  \
178403  Hai Rendi,Nyeri dada seperti yang Anda alami p...   
455608  Salam, Dari info yang Anda sampaikan, setelah ...   
189467  Halo,  Rasa linu / nyeri otot  yang dirasakan ...   
449623  Halo Andini...... Terimakasih atas pertanyaan ...   
108563  Halo,Pterigium merupakan selaput menonjol yang...   

                   predicted_labels  
178403  [1-FR, 3-PI, 4-DM, 5-EDTRB]  
455608        [1-FR, 3-PI, 5-EDTRB]  
189467  [1-FR, 3-PI, 4-DM, 5-EDTRB]  
449623  [1-FR, 3-PI, 4-DM, 5-EDTRB]  
108563        [1-FR, 3-PI, 5-EDTRB]  


In [17]:
sample_df[["answer", "predicted_labels"]].head()

,answer,predicted_labels
178403,"Hai Rendi,Nyeri dada seperti yang Anda alami p...","[1-FR, 3-PI, 4-DM, 5-EDTRB]"
455608,"Salam, Dari info yang Anda sampaikan, setelah ...","[1-FR, 3-PI, 5-EDTRB]"
189467,"Halo, Rasa linu / nyeri otot yang dirasakan ...","[1-FR, 3-PI, 4-DM, 5-EDTRB]"
449623,Halo Andini...... Terimakasih atas pertanyaan ...,"[1-FR, 3-PI, 4-DM, 5-EDTRB]"
108563,"Halo,Pterigium merupakan selaput menonjol yang...","[1-FR, 3-PI, 5-EDTRB]"


In [18]:
sample_df.to_csv("../results/sample_predictions.csv", index=False)

### Test Prediksi

In [ ]:
def predict_text(text):
    text = text_clean_pipeline(text)
    tfidf = best_vectorizer.transform([text])
    pred = best_model.predict(tfidf)[0]

    result = [LABEL_COLS[i] for i in range(len(pred)) if pred[i] == 1]
    return result

print(predict_text("Sebaiknya Anda istirahat dan minum air putih"))

['1-FR', '3-PI', '5-EDTRB']
